# 04e｜Kaggle 轻量承接已有金融模型并训练 SFT

本页使用从 Colab 导出的私有轻量包，承接 `domain_v1` 已训练权重与原词表。
不需要1.54 GiB的完整转运包。第一次只迁移模型权重，重新建立优化器；
不是跨平台精确恢复旧优化器。原模型与标签包属于私人材料：请使用 Private 数据集和 Notebook。

先在 Kaggle 添加轻量私有数据集；选择 GPU T4 x2 和 Internet on。
实际只使用第一张卡。按顺序运行，并在配置格填写轻量包对应 SHA256。
轻量包保留 Colab 导出时核查来源的报告，但 Kaggle 无法重新读取未转运的原始清单。
训练结果是标题字面实体立场的探索性试验，非证券预测或 X 场景验证。

In [ ]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # 首次解析并显示提交；复现时替换成完整的 40 位提交 SHA。
PROJECT = Path("/kaggle/working/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF 不能唯一解析；请填写完整提交 SHA。")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/kaggle/working/130U-FinLLM 已属于其他仓库，请选择新目录。")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("本次固定代码提交：", REVISION)

# 保留 Colab 自带的 CUDA / PyTorch；此处仅安装项目的其他依赖。
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} 与现有环境不兼容：{incompatible}")
    print("tokenizers 兼容性检查通过：", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"缺少 {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} 不满足 {requirement}")
    if problems:
        raise RuntimeError("项目依赖检查失败：" + "; ".join(problems))
    print("项目已安装依赖检查通过。")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"缺少项目 Python 源码：{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"当前 Notebook 导入了错误的项目位置：{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("当前 Notebook 的检查点恢复接口不可用。")
    print("当前 Notebook 内核导入检查通过：", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("新 Python 进程导入检查通过：", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("需要 PyTorch >= 2.5；请更换较新的 Colab 运行环境。")
tokenizer_requirements = tokenizer_install_requirements()
print("本次 tokenizers 约束：", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("PyTorch 环境意外变化，请保留输出并停止训练。")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook 环境准备完成；PyTorch 保持：", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("缺少阶段输入，请先准备或从 Drive 恢复：\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/kaggle/working")
print(f"/kaggle/working 剩余磁盘：{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
from finllm.kaggle_lite import install_lite_archive, install_lite_directory, prepare_lite_sft
from finllm.training import run_training
from finllm.checkpoint import load_checkpoint, sha256_file
from finllm.runtime import restore_seed_checkpoint
from finllm.evaluation import run_evaluation
import torch

# 从轻量导出报告复制清单 SHA256；恢复时使用相同输入与代码版本。
LITE_MANIFEST_SHA256 = "请填轻量包manifest_sha256".strip().lower()
# 仅当 Kaggle 没有自动解压 ZIP 时，才需填写 ZIP 的 SHA256。
LITE_ZIP_SHA256 = "".strip().lower()
INPUT = Path("/kaggle/input")
LITE = Path("/kaggle/working/130u_lite_input")
RUN = Path("/kaggle/working/130u_lite_sft_v1")
manifests = sorted(INPUT.rglob("lite_manifest.json"))
archives = sorted(INPUT.rglob("130u_kaggle_lite.zip"))
if len(manifests) == 1:
    installed = install_lite_directory(manifests[0].parent, LITE,
                                      expected_manifest_sha256=LITE_MANIFEST_SHA256)
elif not manifests and len(archives) == 1:
    installed = install_lite_archive(archives[0], LITE, expected_zip_sha256=LITE_ZIP_SHA256)
else:
    raise RuntimeError("请用 Add Input 添加唯一的 Private 轻量数据集。")
print("轻量输入核验：", json.dumps(installed, indent=2, ensure_ascii=False))

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("请选择 T4 GPU，并从第一格重新运行。")
print("GPU数量：", torch.cuda.device_count(), "本次使用：", torch.cuda.get_device_name(0))
pilot = prepare_lite_sft(PROJECT, LITE, RUN,
                         expected_manifest_sha256=installed["manifest_sha256"], epochs=2)
config = pilot["config"]
print("任务分区：", json.dumps(pilot["counts"], indent=2, ensure_ascii=False))
latest = restore_seed_checkpoint(RUN, Path(config["backup_dir"]))
if latest:
    previous = load_checkpoint(latest, allow_previous=False)
    if previous.get("config", {}).get("lite_binding") != config.get("lite_binding"):
        raise ValueError("现有检查点与轻量输入身份不同；保留原实验并停止。")
    config.pop("init_checkpoint", None)
    config["resume"] = str(latest)
    print("检测到已有检查点，将按相同运行环境严格恢复。")
result = run_training(config)
print("SFT训练结果：", json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
for name, checkpoint in (("domain_before", LITE / "weights/domain_weights.pt"),
                         ("sft_best", RUN / "best.pt")):
    metrics = run_evaluation(checkpoint, config["tokenizer"], config["validation_data"],
                             RUN / (name + "_development.json"), target_mode="literal_entity")
    print(name, "开发集结果（探索性，不是最终测试）：", json.dumps(metrics, indent=2, ensure_ascii=False))
print("模型及训练状态：", RUN)
print("词表及迁移输入：", LITE)
print("恢复检查点 SHA256：", sha256_file(RUN / "latest.pt"))
print("恢复完整性标记 SHA256：", sha256_file(RUN / "latest.pt.json"))
print("请保存 Private Notebook 版本和输出，或下载 latest.pt 及 latest.pt.json；")
print("同时保留轻量 Private 数据集和第一格代码版本，工作目录不保证永久保存。")

如果中途断线，同一会话保留文件时从第一格重跑。跨会话恢复须先保留并恢复原工作目录，
固定第一格 `REF` 为首次运行的完整40位提交，且设备、PyTorch/CUDA、数据和路径一致。
旧 Drive 权重不会被修改。开发集指标不是最终评估，原发布者内容权利仍需单独核验。